# EEG publication analysis workspace

This notebook is the human-facing companion to the frozen EEG preprocessing pipeline. It is designed for inspection, figure review, exploratory play, and manuscript development without mutating Silver or Gold data.

## Questions

1. Do sustained EEG features differ between any-ad and no-ad conversations, inline and labelled-block formats, or early and late timing?
2. Do ad-locked post-onset changes differ from timing-matched no-ad replies?
3. Are conclusions stable across the 1,000, 1,050, and 1,500 µV artifact policies?
4. Which participant-level patterns and uncertainty ranges should drive interpretation?

## Analysis hierarchy

- **Primary:** Fz theta and posterior alpha with prespecified participant-level contrasts.
- **Secondary:** global theta/alpha/beta, FAA, and factorial interactions.
- **Exploratory:** engagement ratios, delta/gamma, and relative powers.

The participant is the replication unit. A non-significant result is not evidence of absence.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# Resolve the repository whether the notebook starts from its own folder or repo root.
START = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (START, *START.parents)
    if (path / "analysis/eeg/preprocessing").exists()
)
ANALYSIS_ROOT = REPO_ROOT / "analysis/eeg/analysis"
OUTPUT_ROOT = ANALYSIS_ROOT / "outputs"
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from run_publication_analysis import run

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
plt.rcParams["figure.dpi"] = 120
print(f"Repository: {REPO_ROOT}")
print(f"Analysis workspace: {ANALYSIS_ROOT}")

## 1. Reproduce the complete analysis

This calls the production runner. It reads immutable Gold and statistics inputs, regenerates all tables and figures, hashes the inputs, and fails if any of twenty quality gates fails.

In [ ]:
manifest = run(
    ANALYSIS_ROOT / "analysis_config.json",
    OUTPUT_ROOT,
)
print(manifest["status"])
print(f"Quality gates: {manifest['quality_gates']['passed']}/{manifest['quality_gates']['total']}")

In [ ]:
quality = json.loads((OUTPUT_ROOT / "reports/quality_gates.json").read_text())
quality_table = pd.DataFrame(
    [{"quality_gate": key, "passed": value} for key, value in quality["checks"].items()]
)
display(quality_table)
assert quality["status"] == "pass"

## 2. Inspect the analysis-ready datasets

There are two complementary participant-level datasets:

- `condition_features.csv`: one baseline or sustained-condition window per participant and condition.
- `ad_response_features.csv`: one pre/post response pair per participant and ad or matched no-ad reference.

The larger epoch table is available for artifact and retention diagnostics, but inferential tests use participant-level summaries.

In [ ]:
GOLD = REPO_ROOT / "src/project/logs/xdf/gold/features"
STATS = REPO_ROOT / "analysis/eeg/statistics/outputs"

condition = pd.read_csv(GOLD / "condition_features.csv")
ad_response = pd.read_csv(GOLD / "ad_response_features.csv")
condition_epochs = pd.read_csv(GOLD / "condition_epoch_features.csv")
condition_tests = pd.read_csv(STATS / "eeg_condition_contrasts.csv")
ad_tests = pd.read_csv(STATS / "eeg_ad_response_contrasts.csv")
condition_scores = pd.read_csv(STATS / "eeg_condition_contrast_scores.csv")
ad_scores = pd.read_csv(STATS / "eeg_ad_response_contrast_scores.csv")

summary = pd.DataFrame([
    {"dataset": "condition summaries", "rows": len(condition), "participants": condition.subject_id.nunique(), "grain": "participant × baseline/condition window"},
    {"dataset": "condition epochs", "rows": len(condition_epochs), "participants": condition_epochs.subject_id.nunique(), "grain": "participant × window × 4-second epoch"},
    {"dataset": "ad responses", "rows": len(ad_response), "participants": ad_response.subject_id.nunique(), "grain": "participant × ad/no-ad response pair"},
])
display(summary)
display(condition.head(3))

In [ ]:
# Compact data dictionary for the columns most useful during analysis.
data_dictionary = pd.DataFrame([
    ("subject_id", "Participant identifier; the inferential replication unit"),
    ("condition", "No-ad, inline/block, and early/late condition cell"),
    ("retained_epoch_fraction", "Fraction of complete 4-second epochs retained"),
    ("primary_analysis_eligible", "Frozen eligibility decision"),
    ("fz_theta_power_db_uv2_median", "Median Fz theta power within a sustained window"),
    ("posterior_alpha_power_db_uv2_median", "Median posterior alpha across O/P region"),
    ("*_post_minus_pre", "Ad-locked post-onset minus pre-onset response"),
    ("faa_log_f4_minus_f3", "Log alpha F4 minus log alpha F3"),
    ("engagement_*", "Exploratory ratio features; not direct psychological measures"),
], columns=["field", "meaning"])
display(data_dictionary)

## 3. Cohort and retention

The flow figure deliberately separates participants, sustained windows, four-second epochs, and ad/no-ad response pairs because these counts are not interchangeable.

In [ ]:
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_01_data_flow.png")))
qc = pd.read_csv(OUTPUT_ROOT / "tables/cohort_qc_summary.csv")
display(qc)

## 4. Primary sustained-condition analysis

Three contrasts are tested separately for Fz theta and posterior alpha. Holm correction is applied across the three contrasts within each feature. A second, more conservative column corrects all six primary tests together. Bootstrap intervals and leave-one-participant-out stability accompany the parametric estimates. The uncontrolled baseline is not part of these confirmatory tests.

In [ ]:
primary_condition = pd.read_csv(OUTPUT_ROOT / "tables/primary_condition_results.csv")
columns = [
    "contrast_label", "feature_label", "n_participants", "mean_difference",
    "ci_lower", "ci_upper", "bootstrap_mean_ci_lower", "bootstrap_mean_ci_upper",
    "cohen_dz", "p_t_raw", "p_t_holm", "p_t_holm_global_primary_family",
    "p_wilcoxon_holm", "leave_one_out_sign_stability", "holm_significant",
]
display(primary_condition[columns].round(4))
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_02_condition_primary_forest.png")))
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_04_condition_profiles.png")))

## 5. Primary ad-locked response analysis

Each ad cell is compared with a no-ad reply projected at the same early or late turn. The response is `post − pre` over adjacent four-second epochs. Holm correction is applied across four ad-versus-control contrasts within each feature; a global eight-test Holm column is included as a conservative sensitivity.

In [ ]:
primary_ad = pd.read_csv(OUTPUT_ROOT / "tables/primary_ad_results.csv")
display(primary_ad[columns].round(4))
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_03_ad_primary_forest.png")))
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_05_participant_distributions.png")))

## 6. Assumption and influence diagnostics

With only 18 participants, diagnostics are descriptive rather than mechanical pass/fail filters. Compare the t-test with Wilcoxon results, inspect skew and outliers, and emphasize leave-one-participant-out sign stability. Do not delete a participant solely because a normality test is significant.

In [ ]:
diagnostics = pd.read_csv(OUTPUT_ROOT / "tables/assumption_influence_diagnostics.csv")
display(
    diagnostics.sort_values(
        ["leave_one_out_sign_stability", "shapiro_p"]
    ).round(4)
)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_data = diagnostics.copy()
labels = plot_data["analysis"] + ": " + plot_data["contrast_id"] + " / " + plot_data["feature_label"]
ax.scatter(plot_data["leave_one_out_sign_stability"], plot_data["maximum_leave_one_out_mean_shift"], color="#355C7D")
for x, y, label in zip(plot_data["leave_one_out_sign_stability"], plot_data["maximum_leave_one_out_mean_shift"], labels):
    if x < 1.0:
        ax.annotate(label, (x, y), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Leave-one-participant-out sign stability")
ax.set_ylabel("Maximum mean shift after deleting one participant")
ax.set_title("Primary-feature influence diagnostics")
plt.show()

## 7. Artifact-threshold sensitivity

The 1,050 µV primary cutoff was selected after observing Subject 14's 1,042.56 µV epoch. It is therefore post-review and participant-retention-driven. The 1,000 and 1,500 µV branches are mandatory, not optional.

In [ ]:
sensitivity = pd.read_csv(OUTPUT_ROOT / "tables/threshold_sensitivity_results.csv")
display(Image(filename=str(OUTPUT_ROOT / "figures/figure_06_threshold_sensitivity.png")))

# Estimate displacement relative to the 1,050 µV primary policy.
index_columns = ["analysis", "contrast_id", "feature"]
wide = sensitivity.pivot_table(index=index_columns, columns="policy", values="mean_difference")
wide["1000_minus_1050"] = wide["frozen_v1_1000_uv"] - wide["frozen_v3_1050_uv"]
wide["1500_minus_1050"] = wide["frozen_v2_1500_uv"] - wide["frozen_v3_1050_uv"]
display(wide.reset_index().round(4))

## 8. Subject 14 and onset-provenance sensitivities

Subject 14 is shown explicitly because the primary 1,050 µV policy was selected just above one known epoch. The onset table separately identifies observed and reconstructed ad timings; reconstructed inline onsets support four-second spectra but not ERP-latency claims.

In [ ]:
subject14 = pd.read_csv(OUTPUT_ROOT / "tables/subject14_influence.csv")
onset_provenance = pd.read_csv(OUTPUT_ROOT / "tables/ad_onset_provenance.csv")

display(Markdown("### Leave-Subject-14-out primary-feature contrasts"))
display(subject14.round(4))
display(Markdown("### Ad/no-ad onset provenance"))
display(onset_provenance.round(6))

## 9. Interactive feature sandbox

Change the two variables below to inspect participant-level scores. This sandbox does not change the prespecified hierarchy. Engagement ratios remain exploratory and should be interpreted alongside behavioral measures, not as direct measures of psychological engagement.

In [ ]:
# Edit these values and rerun the cell.
ANALYSIS_TO_EXPLORE = "condition"  # "condition" or "ad"
FEATURE_TO_EXPLORE = "engagement_pope_frontocentral_beta_over_alpha_theta"

score_source = condition_scores if ANALYSIS_TO_EXPLORE == "condition" else ad_scores
selected = score_source[score_source["feature"].eq(FEATURE_TO_EXPLORE)].copy()
if selected.empty:
    available = sorted(score_source["feature"].unique())
    raise ValueError(f"Unknown feature. Choose one of: {available}")

contrast_order = list(selected["contrast_id"].drop_duplicates())
fig, ax = plt.subplots(figsize=(max(8, len(contrast_order) * 1.25), 4.8))
rng = np.random.default_rng(20260803)
for index, contrast_id in enumerate(contrast_order):
    values = selected.loc[selected["contrast_id"].eq(contrast_id), "difference"].to_numpy()
    ax.scatter(np.full(len(values), index) + rng.normal(0, 0.045, len(values)), values, alpha=0.7)
    ax.plot([index - 0.2, index + 0.2], [values.mean(), values.mean()], color="#8A2D2D", linewidth=2)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(contrast_order)), contrast_order, rotation=35, ha="right")
ax.set_ylabel("Participant contrast")
ax.set_title(f"{ANALYSIS_TO_EXPLORE}: {FEATURE_TO_EXPLORE}")
plt.tight_layout()
plt.show()

display(selected.groupby("contrast_id")["difference"].describe().round(4))

## 10. Initial interpretation

The generated narrative is intentionally conservative. It reports compatible effect ranges and robustness without converting non-significance into evidence that advertisements have no EEG effect.

In [ ]:
display(Markdown((OUTPUT_ROOT / "reports/initial_results_summary.md").read_text()))

## 11. Planned behavioral–EEG integration

Do not join behavioral outcomes ad hoc. First freeze a behavioral contract containing:

- one stable participant identifier mapped to `subject_id`;
- condition and ad-reference identifiers at compatible grain;
- prespecified outcomes such as trust, convincingness, helpfulness, relevance, manipulation, recall, and response latency;
- missingness and exclusion fields;
- explicit direction coding for every scale.

Then create a separate multimodal table at participant × condition or participant × ad-response grain. Candidate models should use participant random intercepts or participant-level contrasts, control multiplicity, and keep engagement indices exploratory.

## Remaining publication gates

Closed 2026-08-19: filtering/interpolation figures, ICA component exclusions, and ICA as the primary branch (no-ICA remains the mandatory sensitivity).

Still open:

1. Keep all three engagement metrics and decide their final inferential tier.
2. Optionally verify the reported Cz reference/Fpz ground handling from the acquisition workspace.
3. Implement the era-aware read-versus-write positive control before interpreting null condition effects as evidence of pipeline sensitivity.
4. Freeze behavioral schemas before multimodal integration.
5. Convert these initial figures and tables into manuscript-specific numbering and captions only after scientific review.

In [ ]:
# Reproducibility record: every source path and SHA-256 hash used by the runner.
manifest_view = json.loads((OUTPUT_ROOT / "reports/analysis_manifest.json").read_text())
inputs = pd.DataFrame([
    {"input": name, **details} for name, details in manifest_view["inputs"].items()
])
display(inputs)
print("Human gates:")
for gate in manifest_view["human_gates"]:
    print(f"- {gate}")